# Week 7 - Document Question Answering System (RAG)

Building a simple RAG pipeline that answers questions from my resume using retrieval + generation instead of relying on the model's own knowledge.

## Setup

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/misthijain/resume-misthijain/Resume_MisthiJain.pdf


In [3]:
!pip install -q sentence-transformers faiss-cpu PyPDF2 transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 83.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.7 MB/s eta 0:00:00


In [4]:
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import PyPDF2

## Step 1 - Loading the Document

Reading my resume PDF and pulling out the raw text.

In [10]:
def load_pdf(path):
    text = ""
    reader = PyPDF2.PdfReader(path)
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

In [17]:
pdf_path = "/kaggle/input/datasets/misthijain/resume-misthijain/Resume_MisthiJain.pdf"
raw_text = load_pdf(pdf_path)
print(raw_text[:500])

MISTHI JAIN
SUMMARY
IT undergraduate at Jaypee Institute of Information Technology with a strong foundation in data
structures, algorithms, and software development. Skilled in problem-solving and building efficient
solutions using modern programming concepts. Actively enhancing technical expertise through hands-
on projects, with an additional interest in graphic design.
Noida, Uttar Pradesh | 6395031542 | misthijain2006@gmail.com 
LinkedIn: linkedin.com/in/misthi-jain
EDUCATION
Bachelor of Tec


## Step 2 - Chunking the Text

Splitting the resume into small overlapping chunks so retrieval can find specific sections instead of searching the whole document at once.

In [20]:
def chunk_text(text, chunk_size=40, overlap=8):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [21]:
chunks = chunk_text(raw_text)
print(f"Total chunks: {len(chunks)}")

for i, c in enumerate(chunks):
    print(f"Chunk {i}: {c[:80]}...")
    

Total chunks: 11
Chunk 0: MISTHI JAIN SUMMARY IT undergraduate at Jaypee Institute of Information Technolo...
Chunk 1: concepts. Actively enhancing technical expertise through hands- on projects, wit...
Chunk 2: XII (CBSE) Jaypee Institute of Information Technology, Noida 2024-2028 | CGPA: 6...
Chunk 3: move validation and check detection for all 6 piece types Applied OOP principles...
Chunk 4: updates using 2D array representation TECHNICAL SKILLS Languages: C++, Java, Pyt...
Chunk 5: Git, GitHub, VS Code, CanvaClass X (CBSE) Agra Public School, Agra 2022 | 96.6% ...
Chunk 6: session handling Implemented real-time buy/sell orders, live price updates, and ...
Chunk 7: Pandas, NumPy Built a content-based movie recommendation engine on a real-world ...
Chunk 8: and encoding to prepare model input pipeline Analyzed model limitations and accu...
Chunk 9: ML challenges including data sparsity and feature relevance TRAINING Graphic Des...
Chunk 10: Canva Built foundational skills in visual c

## Step 3 - Creating Embeddings

Converting each chunk into a vector using a pretrained sentence embedding model.

In [22]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True)
embedding_dim = chunk_embeddings.shape[1]
print(f"Embedding dim: {embedding_dim}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dim: 384


## Step 4 - Building the Vector Store

Storing all chunk embeddings in a FAISS index for fast similarity search.

In [24]:
index = faiss.IndexFlatL2(embedding_dim)
index.add(np.array(chunk_embeddings).astype("float32"))
print(f"Vectors stored: {index.ntotal}")

Vectors stored: 11


## Step 5 & 6 - Query Processing and Retrieval

Turning a question into an embedding, then searching the vector store for the closest matching chunks.

In [25]:
def retrieve_chunks(query, top_k=3):
    query_embedding = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    retrieved = [(chunks[i], distances[0][j]) for j, i in enumerate(indices[0])]
    return retrieved

In [26]:
test = retrieve_chunks("What is Misthi's CGPA?")
for chunk, dist in test:
    print(f"Distance: {dist:.4f} | Chunk: {chunk[:100]}...")

Distance: 1.4210 | Chunk: concepts. Actively enhancing technical expertise through hands- on projects, with an additional inte...
Distance: 1.4452 | Chunk: MISTHI JAIN SUMMARY IT undergraduate at Jaypee Institute of Information Technology with a strong fou...
Distance: 1.6730 | Chunk: XII (CBSE) Jaypee Institute of Information Technology, Noida 2024-2028 | CGPA: 6.7/10 Agra Public Sc...


## Step 7 - Answer Generation

Feeding the retrieved chunks and question into a language model to generate a grounded answer.

In [29]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_answer(query, retrieved_chunks):
    context = "\n".join([c[0] for c in retrieved_chunks])
    prompt = f"Answer the question based only on the context below.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_length=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [30]:
def rag_pipeline(query, top_k=3):
    retrieved = retrieve_chunks(query, top_k)
    answer = generate_answer(query, retrieved)
    return answer, retrieved

In [31]:
answer, retrieved = rag_pipeline("What is Misthi's CGPA?")
print(answer)

6.7/10


## Testing the Pipeline

Asking a few questions about my resume and checking if the answers are actually correct.

In [32]:
test_questions = [
    "What projects has Misthi built?",
    "What programming languages does Misthi know?",
    "What ML models were used in the recommendation system project?",
    "What is Misthi's CGPA?"
]

for q in test_questions:
    answer, retrieved = rag_pipeline(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 50)

Q: What projects has Misthi built?
A: UI/UX design
--------------------------------------------------
Q: What programming languages does Misthi know?
A: C++, Java, Python, JavaScript
--------------------------------------------------
Q: What ML models were used in the recommendation system project?
A: KNN, Decision Tree, Random Forest, and SVM
--------------------------------------------------
Q: What is Misthi's CGPA?
A: 6.7/10
--------------------------------------------------


## Validation Log

Showing which chunks got retrieved for each question along with distance scores, so retrieval accuracy is visible.

In [34]:
print("Validation Log")
for q in test_questions:
    retrieved = retrieve_chunks(q)
    for chunk, dist in retrieved:
        print(f"Query: {q} | Distance: {dist:.4f} | Chunk: {chunk[:80]}...")

Validation Log
Query: What projects has Misthi built? | Distance: 1.2148 | Chunk: concepts. Actively enhancing technical expertise through hands- on projects, wit...
Query: What projects has Misthi built? | Distance: 1.2618 | Chunk: MISTHI JAIN SUMMARY IT undergraduate at Jaypee Institute of Information Technolo...
Query: What projects has Misthi built? | Distance: 1.5585 | Chunk: Canva Built foundational skills in visual communication applicable to UI/UX desi...
Query: What programming languages does Misthi know? | Distance: 1.1668 | Chunk: MISTHI JAIN SUMMARY IT undergraduate at Jaypee Institute of Information Technolo...
Query: What programming languages does Misthi know? | Distance: 1.2948 | Chunk: concepts. Actively enhancing technical expertise through hands- on projects, wit...
Query: What programming languages does Misthi know? | Distance: 1.3020 | Chunk: updates using 2D array representation TECHNICAL SKILLS Languages: C++, Java, Pyt...
Query: What ML models were used in the r

## System Metrics Report

Summary of the setup used for this pipeline.

In [35]:
print("System Metrics Report")
print("Document: Resume_MisthiJain.pdf")
print(f"Chunking: chunk_size=40 words, overlap=8 words")
print(f"Total chunks: {len(chunks)}")
print(f"Embedding model: all-MiniLM-L6-v2, dim: {embedding_dim}")
print("Vector store: FAISS IndexFlatL2")
print("Generation model: google/flan-t5-base")

System Metrics Report
Document: Resume_MisthiJain.pdf
Chunking: chunk_size=40 words, overlap=8 words
Total chunks: 11
Embedding model: all-MiniLM-L6-v2, dim: 384
Vector store: FAISS IndexFlatL2
Generation model: google/flan-t5-base


## Experiment - Changing Chunk Size

Trying a bigger chunk size to see if retrieval quality changes.

In [36]:
chunks_big = chunk_text(raw_text, chunk_size=60, overlap=10)
print(f"Total chunks with bigger size: {len(chunks_big)}")

big_embeddings = embed_model.encode(chunks_big, show_progress_bar=True)
big_index = faiss.IndexFlatL2(big_embeddings.shape[1])
big_index.add(np.array(big_embeddings).astype("float32"))

def retrieve_chunks_big(query, top_k=3):
    query_embedding = embed_model.encode([query]).astype("float32")
    distances, indices = big_index.search(query_embedding, top_k)
    return [(chunks_big[i], distances[0][j]) for j, i in enumerate(indices[0])]

sample_q = "What is Misthi's CGPA?"
for chunk, dist in retrieve_chunks_big(sample_q):
    print(f"Distance: {dist:.4f} | Chunk: {chunk[:100]}...")

Total chunks with bigger size: 7


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Distance: 1.3226 | Chunk: MISTHI JAIN SUMMARY IT undergraduate at Jaypee Institute of Information Technology with a strong fou...
Distance: 1.3424 | Chunk: Pradesh | 6395031542 | misthijain2006@gmail.com LinkedIn: linkedin.com/in/misthi-jain EDUCATION Bach...
Distance: 1.8690 | Chunk: OOP, DBMS Technologies: Socket Programming, JDBC, MySQL Tools & Platforms: Git, GitHub, VS Code, Can...


## Observation

- With chunk_size=40, retrieval was more focused per section since fewer topics got mixed into one chunk
- With chunk_size=60, chunk count dropped from 11 to 7, and sections got blended together more
- For the CGPA question specifically, chunk_size=40 correctly retrieved the education chunk containing "CGPA: 6.7/10", while chunk_size=60 failed to retrieve it in the top 3 results at all
- The question "What projects has Misthi built?" retrieved the wrong chunks (Summary and Training sections) instead of the actual Projects section, because those chunks also contained the word "projects" or related terms — showing a real limitation of small-scale semantic retrieval on short documents
- Smaller chunk size worked better overall for a short, structured document like a resume